# 01 — Exploratory Data Analysis

Análise exploratória com dois snapshots do Inside Airbnb (Rio de Janeiro):
- **Junho/2025** e **Setembro/2025** — comparação entre snapshots
- Distribuição e evolução de preços
- Análise por bairro e tipo de acomodação
- Sazonalidade via calendário combinado
- Impacto de amenities no preço

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

RAW_PATH = Path('../data/raw')
PROCESSED_PATH = Path('../data/processed')
PROCESSED_PATH.mkdir(exist_ok=True)

SNAPSHOTS = {
    'jun/2025': 'rio-de-janeiro_listings_2025-06-24.csv.gz',
    'set/2025': 'rio-de-janeiro_listings_2025-09-26.csv.gz',
}
CALENDAR_FILES = {
    'jun/2025': 'rio-de-janeiro_calendar_2025-06-24.csv.gz',
    'set/2025': 'rio-de-janeiro_calendar_2025-09-26.csv.gz',
}

## 1. Carregamento dos dados

In [ ]:
def load_listings(filename):
    df = pd.read_csv(RAW_PATH / filename, compression='gzip', low_memory=False)
    df['price_num'] = (
        df['price'].astype(str)
        .str.replace(r'[$,]', '', regex=True)
        .astype(float)
    )
    df = df[(df['price_num'] >= 10) & (df['price_num'] <= 5000)].copy()
    df['log_price'] = np.log1p(df['price_num'])
    return df

dfs = {label: load_listings(f) for label, f in SNAPSHOTS.items()}

for label, df in dfs.items():
    print(f"{label}: {df.shape[0]:,} listings | preço mediano: R${df['price_num'].median():.0f}")

In [ ]:
# Colunas com alto % de nulos (setembro)
df_set = dfs['set/2025']
null_pct = (df_set.isnull().sum() / len(df_set) * 100).sort_values(ascending=False)
print('Colunas com > 20% de nulos (set/2025):')
print(null_pct[null_pct > 20].to_string())

## 2. Comparação entre Snapshots

In [ ]:
# Listings novos, removidos e em comum
ids_jun = set(dfs['jun/2025']['id'])
ids_set = set(dfs['set/2025']['id'])

novos    = ids_set - ids_jun
removidos = ids_jun - ids_set
comuns   = ids_jun & ids_set

print(f"Listings em junho/2025:    {len(ids_jun):,}")
print(f"Listings em setembro/2025: {len(ids_set):,}")
print(f"Novos em setembro:         {len(novos):,} (+{len(novos)/len(ids_jun)*100:.1f}%)")
print(f"Removidos desde junho:     {len(removidos):,} (-{len(removidos)/len(ids_jun)*100:.1f}%)")
print(f"Em comum:                  {len(comuns):,}")

In [ ]:
# Evolução de preço nos listings em comum
df_jun = dfs['jun/2025'].set_index('id')
df_set = dfs['set/2025'].set_index('id')
common_ids = list(comuns)

price_change = pd.DataFrame({
    'price_jun': df_jun.loc[common_ids, 'price_num'],
    'price_set': df_set.loc[common_ids, 'price_num'],
}).dropna()

price_change['delta_pct'] = (price_change['price_set'] / price_change['price_jun'] - 1) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(price_change['price_jun'], price_change['price_set'],
                alpha=0.2, s=5, color='steelblue')
lim = [0, price_change[['price_jun', 'price_set']].max().max()]
axes[0].plot(lim, lim, 'r--', lw=1)
axes[0].set_xlabel('Preço Jun/2025 (R$)')
axes[0].set_ylabel('Preço Set/2025 (R$)')
axes[0].set_title('Preço: Junho vs Setembro (listings em comum)')

axes[1].hist(price_change['delta_pct'].clip(-80, 80), bins=60,
             color='teal', edgecolor='white')
axes[1].axvline(0, color='red', lw=1, linestyle='--')
axes[1].axvline(price_change['delta_pct'].median(), color='orange', lw=1.5,
                linestyle='--', label=f"Mediana: {price_change['delta_pct'].median():.1f}%")
axes[1].set_xlabel('Variação de preço (%)')
axes[1].set_title('Distribuição da Variação de Preço')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Variação mediana de preço: {price_change['delta_pct'].median():.1f}%")
print(f"Listings com aumento > 10%: {(price_change['delta_pct'] > 10).sum():,}")
print(f"Listings com queda > 10%:   {(price_change['delta_pct'] < -10).sum():,}")

## 3. Distribuição de Preços

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'teal']

for ax, (label, df), color in zip(axes, dfs.items(), colors):
    ax.hist(df['price_num'], bins=80, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(df['price_num'].median(), color='red', linestyle='--',
               label=f"Mediana: R${df['price_num'].median():.0f}")
    ax.set_title(f'Distribuição de Preço — {label}')
    ax.set_xlabel('Preço por noite (R$)')
    ax.legend()

plt.tight_layout()
plt.show()

# Estatísticas comparadas
stats = pd.DataFrame({
    label: df['price_num'].describe()
    for label, df in dfs.items()
})
print(stats.round(1))

In [ ]:
# Distribuição do log(price) — os dois sobrepostos
fig, ax = plt.subplots(figsize=(10, 4))
for (label, df), color in zip(dfs.items(), ['steelblue', 'teal']):
    ax.hist(df['log_price'], bins=80, alpha=0.6, color=color, edgecolor='white', label=label)
ax.set_title('Distribuição de log(Preço) — Junho vs Setembro')
ax.set_xlabel('log1p(Preço)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Preço por tipo de quarto
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, df) in zip(axes, dfs.items()):
    order = df.groupby('room_type')['price_num'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x='room_type', y='price_num', order=order, ax=ax, showfliers=False)
    ax.set_title(f'Preço por Tipo — {label}')
    ax.set_xlabel('')
    ax.set_ylabel('R$')
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 4. Análise por Bairro

In [ ]:
# Usar setembro para análise principal
df = dfs['set/2025']

neighbourhood_stats = (
    df.groupby('neighbourhood_cleansed')['price_num']
    .agg(['median', 'count'])
    .query('count >= 30')
    .sort_values('median', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(neighbourhood_stats.index, neighbourhood_stats['median'], color='steelblue')
ax.set_xlabel('Preço mediano por noite (R$)')
ax.set_title('Top 20 Bairros — Preço Mediano (Set/2025)')
ax.invert_yaxis()
for bar, (_, row) in zip(bars, neighbourhood_stats.iterrows()):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'n={row["count"]:.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Variação de preço mediano por bairro entre os dois snapshots
bairro_jun = dfs['jun/2025'].groupby('neighbourhood_cleansed')['price_num'].median()
bairro_set = dfs['set/2025'].groupby('neighbourhood_cleansed')['price_num'].median()

bairro_delta = pd.DataFrame({'jun': bairro_jun, 'set': bairro_set}).dropna()
bairro_delta['delta_pct'] = (bairro_delta['set'] / bairro_delta['jun'] - 1) * 100
bairro_delta = bairro_delta.sort_values('delta_pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['green' if v > 0 else 'red' for v in bairro_delta['delta_pct']]
ax.barh(bairro_delta.index, bairro_delta['delta_pct'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Variação de Preço por Bairro: Jun → Set/2025 (%)')
ax.set_xlabel('% variação no preço mediano')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Correlações com Preço

In [ ]:
df = dfs['set/2025']

numeric_cols = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'number_of_reviews', 'review_scores_rating',
    'review_scores_cleanliness', 'review_scores_location',
    'calculated_host_listings_count', 'availability_365',
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

corr_data = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
corr_data['log_price'] = df['log_price']
corr = corr_data.corr()['log_price'].drop('log_price').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['green' if v > 0 else 'red' for v in corr]
corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlação com log(Preço) — Set/2025')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()
print(corr.to_string())

## 6. Sazonalidade via Calendário (Junho + Setembro)

In [ ]:
# Carregar e combinar os dois calendários
def load_calendar(filename):
    df = pd.read_csv(RAW_PATH / filename, compression='gzip', parse_dates=['date'])
    df['price_num'] = (
        df['price'].astype(str)
        .str.replace(r'[$,]', '', regex=True)
        .astype(float)
    )
    return df.dropna(subset=['price_num'])

cal_jun = load_calendar(CALENDAR_FILES['jun/2025'])
cal_set = load_calendar(CALENDAR_FILES['set/2025'])

# Combinar sem duplicatas
cal_combined = pd.concat([cal_jun, cal_set]).drop_duplicates(
    subset=['listing_id', 'date']
).sort_values('date')

print(f"Calendário junho:    {len(cal_jun):,} linhas | datas: {cal_jun['date'].min().date()} → {cal_jun['date'].max().date()}")
print(f"Calendário setembro: {len(cal_set):,} linhas | datas: {cal_set['date'].min().date()} → {cal_set['date'].max().date()}")
print(f"Combinado:           {len(cal_combined):,} linhas")

In [ ]:
# Série temporal de preço mediano diário
daily = cal_combined.groupby('date')['price_num'].median().sort_index()
daily = daily.dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(daily.index, daily.values, linewidth=0.8, color='steelblue')
axes[0].set_title('Preço Mediano Diário — Calendário Combinado (Jun + Set/2025)')
axes[0].set_ylabel('R$')

dow = daily.groupby(daily.index.dayofweek).mean()
dow.index = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
axes[1].bar(dow.index, dow.values, color='teal')
axes[1].set_title('Preço Médio por Dia da Semana')
axes[1].set_ylabel('R$')

plt.tight_layout()
plt.show()

fim_semana = dow[['Sáb', 'Dom']].mean()
semana = dow[['Seg', 'Ter', 'Qua', 'Qui', 'Sex']].mean()
print(f'Variação fim de semana vs semana: +{(fim_semana/semana - 1)*100:.1f}%')

In [ ]:
# Decomposição STL
from statsmodels.tsa.seasonal import STL

daily_clean = daily.asfreq('D').interpolate()
stl = STL(daily_clean, period=7, robust=True)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
result.observed.plot(ax=axes[0], title='Observado', color='steelblue')
result.trend.plot(ax=axes[1], title='Tendência', color='orange')
result.seasonal.plot(ax=axes[2], title='Sazonalidade (semanal)', color='green')
result.resid.plot(ax=axes[3], title='Resíduo', color='gray')
for ax in axes:
    ax.set_ylabel('R$')
plt.tight_layout()
plt.savefig(PROCESSED_PATH / 'eda_stl_decomposition.png', bbox_inches='tight')
plt.show()

## 7. Análise de Amenities

In [ ]:
def parse_amenities(raw):
    if pd.isna(raw):
        return []
    return [i.lower() for i in re.findall(r'"([^"]+)"', str(raw))]

df = dfs['set/2025'].copy()
df['amenities_list'] = df['amenities'].apply(parse_amenities)

counter = Counter(a for lst in df['amenities_list'] for a in lst)
top_amenities = pd.Series(dict(counter.most_common(25)))

fig, ax = plt.subplots(figsize=(10, 7))
top_amenities.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('25 Amenities Mais Comuns — Set/2025')
ax.set_xlabel('Número de listings')
plt.tight_layout()
plt.show()

In [ ]:
# Impacto de amenities premium no preço
premium = ['pool', 'hot tub', 'gym', 'elevator', 'parking', 'breakfast',
           'fireplace', 'air conditioning', 'washer', 'workspace']

results = []
for amenity in premium:
    has = df['amenities_list'].apply(lambda lst: amenity in lst)
    if has.sum() > 20:
        results.append({
            'amenity': amenity,
            'price_com': df.loc[has, 'price_num'].median(),
            'price_sem': df.loc[~has, 'price_num'].median(),
            'count': has.sum(),
        })

impact = pd.DataFrame(results).set_index('amenity')
impact['premium_pct'] = (impact['price_com'] / impact['price_sem'] - 1) * 100
impact = impact.sort_values('premium_pct', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['green' if v > 0 else 'red' for v in impact['premium_pct']]
impact['premium_pct'].plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Premium de Preço por Amenity (%)')
ax.set_xlabel('% acima da mediana sem a amenity')
plt.tight_layout()
plt.show()
print(impact[['price_com', 'price_sem', 'premium_pct', 'count']].to_string())

## 8. Conclusões do EDA

Preencher após rodar o notebook com os dados reais:

- **Evolução de preços Jun→Set**: +X% de variação mediana
- **Sazonalidade semanal**: fins de semana X% mais caros → Holt-Winters justificado
- **Bairros mais caros**: Ipanema, Leblon, ...
- **Features mais correlacionadas**: accommodates, bedrooms, neighbourhood
- **Amenities com maior impacto**: pool, hot tub, ...
- **Decisão sobre os dois snapshots**: usar setembro para treino, calendário combinado para HW